# Download Data OSM — Kabupaten Banjarnegara

Notebook ini mendownload **jaringan jalan** dan **sungai/badan air** dari OpenStreetMap menggunakan library `osmnx` dan `geopandas`.

Output: file GeoJSON siap pakai untuk project **Inklusi TPS Spatial Intelligence**.

## 1. Install & Import Libraries

In [ ]:
!pip install osmnx geopandas shapely -q

In [ ]:
import osmnx as ox
import geopandas as gpd
import json
from shapely.geometry import shape, mapping
import warnings
warnings.filterwarnings('ignore')

print(f"OSMnx version: {ox.__version__}")
print(f"GeoPandas version: {gpd.__version__}")

## 2. Definisi Area

Menggunakan nama tempat untuk mendapatkan boundary Kabupaten Banjarnegara.

In [ ]:
# Nama area yang akan didownload
PLACE_NAME = "Kabupaten Banjarnegara, Jawa Tengah, Indonesia"

# Download boundary kabupaten
print(f"Downloading boundary: {PLACE_NAME}...")
boundary = ox.geocode_to_gdf(PLACE_NAME)
print(f"Boundary berhasil didownload.")
print(f"Luas: {boundary.to_crs(epsg=32749).area.values[0] / 1e6:.2f} km²")
boundary.plot(figsize=(8, 8), edgecolor='red', facecolor='lightblue', alpha=0.5)
print(boundary.columns.tolist())

## 3. Download Jaringan Jalan

Download semua tipe jalan dari OSM menggunakan `osmnx.graph_from_place()`.

In [ ]:
# Download jaringan jalan - tipe "drive" untuk jalan yang bisa dilalui kendaraan
print("Downloading jaringan jalan (ini bisa memakan waktu beberapa menit)...")
G = ox.graph_from_place(PLACE_NAME, network_type="drive")
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")

# Convert ke GeoDataFrame
nodes, edges = ox.graph_to_gdfs(G)
print(f"\nJumlah ruas jalan: {len(edges):,}")
print(f"\nKolom edges: {edges.columns.tolist()}")
print(f"\nTipe jalan (highway):")
# highway bisa berupa list, jadi kita flatten dulu
highway_types = edges['highway'].apply(lambda x: x if isinstance(x, str) else x[0] if isinstance(x, list) else str(x))
print(highway_types.value_counts())

In [ ]:
# Visualisasi jaringan jalan
fig, ax = ox.plot_graph(G, figsize=(12, 12), node_size=0, edge_linewidth=0.5,
                        edge_color='steelblue', bgcolor='white', show=True)

In [ ]:
# Siapkan data jalan untuk export
# Pilih kolom yang relevan dan bersihkan
road_cols = ['geometry', 'highway', 'name', 'length', 'maxspeed', 'surface', 'lanes', 'oneway']
available_cols = [c for c in road_cols if c in edges.columns]
roads_gdf = edges[available_cols].copy()

# Flatten list values di kolom highway dan name
for col in ['highway', 'name']:
    if col in roads_gdf.columns:
        roads_gdf[col] = roads_gdf[col].apply(
            lambda x: x[0] if isinstance(x, list) else x
        )

# Reset index (u, v, key -> columns)
roads_gdf = roads_gdf.reset_index(drop=True)

# Pastikan CRS WGS84
roads_gdf = roads_gdf.to_crs(epsg=4326)

print(f"Total ruas jalan: {len(roads_gdf):,}")
print(f"\nKolom: {roads_gdf.columns.tolist()}")
print(f"\nSample data:")
roads_gdf.head()

In [ ]:
# Klasifikasi jalan untuk analisis aksesibilitas
# Mapping highway type ke kelas aksesibilitas
ROAD_CLASS_MAP = {
    # Kelas 1 - Jalan utama (aksesibilitas tinggi)
    'motorway': 1, 'motorway_link': 1,
    'trunk': 1, 'trunk_link': 1,
    'primary': 1, 'primary_link': 1,
    # Kelas 2 - Jalan sekunder
    'secondary': 2, 'secondary_link': 2,
    'tertiary': 2, 'tertiary_link': 2,
    # Kelas 3 - Jalan lokal
    'residential': 3, 'living_street': 3,
    'unclassified': 3,
    # Kelas 4 - Jalan kecil/gang
    'service': 4, 'track': 4,
}

roads_gdf['road_class'] = roads_gdf['highway'].map(ROAD_CLASS_MAP).fillna(4).astype(int)

ROAD_CLASS_LABEL = {
    1: 'Jalan Utama',
    2: 'Jalan Sekunder',
    3: 'Jalan Lokal',
    4: 'Jalan Kecil/Gang',
}
roads_gdf['road_class_label'] = roads_gdf['road_class'].map(ROAD_CLASS_LABEL)

print("Distribusi kelas jalan:")
print(roads_gdf['road_class_label'].value_counts())

In [ ]:
# Export jalan ke GeoJSON
output_roads = "jaringan_jalan_banjarnegara.geojson"
roads_gdf.to_file(output_roads, driver='GeoJSON')
print(f"✅ Jaringan jalan berhasil disimpan ke: {output_roads}")
print(f"   Ukuran: {len(open(output_roads).read()) / 1024 / 1024:.1f} MB")

# Download dari Colab
try:
    from google.colab import files
    files.download(output_roads)
    print("📥 File otomatis terdownload.")
except ImportError:
    print("⚠️  Tidak berjalan di Colab. File tersimpan di direktori kerja.")

## 4. Download Sungai & Badan Air

Download sungai dan badan air dari OSM menggunakan `osmnx.features_from_place()`.

In [ ]:
# Download sungai (waterway)
print("Downloading sungai...")
try:
    rivers = ox.features_from_place(
        PLACE_NAME,
        tags={'waterway': ['river', 'stream', 'canal', 'drain', 'ditch']}
    )
    print(f"Total fitur sungai: {len(rivers):,}")
    print(f"Tipe waterway:")
    if 'waterway' in rivers.columns:
        print(rivers['waterway'].value_counts())
except Exception as e:
    print(f"Error: {e}")
    rivers = gpd.GeoDataFrame()

In [ ]:
# Download badan air (natural=water)
print("Downloading badan air...")
try:
    waterbodies = ox.features_from_place(
        PLACE_NAME,
        tags={'natural': ['water']}
    )
    print(f"Total fitur badan air: {len(waterbodies):,}")
except Exception as e:
    print(f"Error: {e}")
    waterbodies = gpd.GeoDataFrame()

In [ ]:
# Siapkan data sungai untuk export
if len(rivers) > 0:
    river_cols = ['geometry', 'waterway', 'name']
    available_river_cols = [c for c in river_cols if c in rivers.columns]
    rivers_export = rivers[available_river_cols].copy()
    rivers_export = rivers_export.reset_index(drop=True)
    
    # Filter hanya LineString dan MultiLineString
    rivers_export = rivers_export[
        rivers_export.geometry.type.isin(['LineString', 'MultiLineString'])
    ]
    
    rivers_export = rivers_export.to_crs(epsg=4326)
    
    output_rivers = "sungai_banjarnegara.geojson"
    rivers_export.to_file(output_rivers, driver='GeoJSON')
    print(f"✅ Sungai berhasil disimpan ke: {output_rivers}")
    print(f"   Total fitur: {len(rivers_export):,}")
    print(f"   Ukuran: {len(open(output_rivers).read()) / 1024 / 1024:.1f} MB")
    
    try:
        from google.colab import files
        files.download(output_rivers)
    except ImportError:
        pass
else:
    print("⚠️ Tidak ada data sungai yang ditemukan.")

In [ ]:
# Export badan air
if len(waterbodies) > 0:
    wb_cols = ['geometry', 'natural', 'name', 'water']
    available_wb_cols = [c for c in wb_cols if c in waterbodies.columns]
    wb_export = waterbodies[available_wb_cols].copy()
    wb_export = wb_export.reset_index(drop=True)
    
    # Filter hanya Polygon dan MultiPolygon
    wb_export = wb_export[
        wb_export.geometry.type.isin(['Polygon', 'MultiPolygon'])
    ]
    
    wb_export = wb_export.to_crs(epsg=4326)
    
    output_wb = "badan_air_banjarnegara.geojson"
    wb_export.to_file(output_wb, driver='GeoJSON')
    print(f"✅ Badan air berhasil disimpan ke: {output_wb}")
    print(f"   Total fitur: {len(wb_export):,}")
    print(f"   Ukuran: {len(open(output_wb).read()) / 1024 / 1024:.1f} MB")
    
    try:
        from google.colab import files
        files.download(output_wb)
    except ImportError:
        pass
else:
    print("⚠️ Tidak ada data badan air yang ditemukan.")

## 5. Ringkasan

Setelah menjalankan semua cell di atas, Anda akan mendapat 3 file GeoJSON:

| File | Isi | Simpan di |
|---|---|---|
| `jaringan_jalan_banjarnegara.geojson` | Jaringan jalan + kelas aksesibilitas | `data/INFRASTRUKTUR/` |
| `sungai_banjarnegara.geojson` | Sungai & kanal | `data/LINGKUNGAN/` |
| `badan_air_banjarnegara.geojson` | Danau, waduk, dll. | `data/LINGKUNGAN/` |

**Catatan:** Data OSM bergantung pada kontribusi komunitas. Kelengkapan jalan di daerah pedesaan mungkin tidak 100%.